# 01 · Dataset audit
Inventory of every acquired source: provenance, schema, coverage, resolution, units and licence. Raw files are read directly so the audit reflects what was downloaded.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30, "display.width", 160)

from src.data import load
from src.analysis import descriptive, trends, seasonality, correlations, statistical_tests as st, data_quality as dq, sensitivity
from src.visualization import charts, maps
from src.data.transform import independent_units

wb = descriptive.real_only(load.load_processed("wb_daily"))
cities = load.load_processed("wb_cities")

## Source registry and pipeline manifest
The manifest records SHA-256 hashes of the raw inputs used for the current processed tables.

In [2]:
import json
from src.utils.config import SOURCES
pd.DataFrame(SOURCES).T[['short','value_type','resolution','license','url']]

,short,value_type,resolution,license,url
wb_pm25,World Bank PM2.5 (modelled),modelled,daily,CC BY-SA 4.0,https://datacatalog.worldbank.org/search/datas...
hf_aq,"Kaggle/HF hourly AQ (modelled, probable CAMS)",modelled,hourly,Unknown (not stated by uploader),https://huggingface.co/datasets/jason1966/ahsa...
openmeteo_era5,Open-Meteo ERA5 weather (reanalysis),modelled,daily,CC BY 4.0 (Open-Meteo API data); ERA5 from Cop...,https://open-meteo.com/en/docs/historical-weat...


In [3]:
print(json.dumps(load.load_manifest()['inputs'], indent=2))

{
  "wb_pm25": {
    "file": "data\\raw\\worldbank_pm25\\worldbank_pm25_pakistan_subset.parquet",
    "rows": 948183,
    "sha256": "556ff438c85901dd5445d4afd0597419c860a1b7a117b7c6841a493f0e50382d",
    "archive_sha256": "df4fe7de6bbb2ee3fb98405d046eeeead1022283c63a3a88021f1be163c91212"
  },
  "hf_aq": {
    "file": "data\\raw\\hf_pakistan_aq\\pakistan_air_quality_final_clean.csv",
    "rows": 21840,
    "sha256": "36ff8cbcd23830116cd834366d2b8447dfdac49489c1543fe38ae3751e7b1529"
  },
  "openmeteo_era5": {
    "dir": "data\\raw\\openmeteo_weather",
    "cities": 22,
    "model": "era5",
    "timezone": "Asia/Karachi"
  }
}


## World Bank raw subset (Pakistan rows of the regional archive)

In [4]:
raw = load.load_wb_raw()
print(raw.shape)
raw.head()

(948183, 12)


,wb_region,nam_0,adm0_wb_a3,pm25_iso3,adm1cd_c,geonameid,city,latitude,longitude,population,date,pm25
0,"Middle East, North Africa, Afghanistan & Pakistan",Pakistan,PAK,PAK,PAK007,1108161,Chuhar Jamali,24.3944,67.99298,18721,2018-10-01,53.741930
1,"Middle East, North Africa, Afghanistan & Pakistan",Pakistan,PAK,PAK,PAK007,1108161,Chuhar Jamali,24.3944,67.99298,18721,2018-10-02,52.962121
2,"Middle East, North Africa, Afghanistan & Pakistan",Pakistan,PAK,PAK,PAK007,1108161,Chuhar Jamali,24.3944,67.99298,18721,2018-10-03,57.224922
3,"Middle East, North Africa, Afghanistan & Pakistan",Pakistan,PAK,PAK,PAK007,1108161,Chuhar Jamali,24.3944,67.99298,18721,2018-10-04,43.554630
4,"Middle East, North Africa, Afghanistan & Pakistan",Pakistan,PAK,PAK,PAK007,1108161,Chuhar Jamali,24.3944,67.99298,18721,2018-10-05,52.801406


In [5]:
raw.dtypes.to_frame('dtype').T

,wb_region,nam_0,adm0_wb_a3,pm25_iso3,adm1cd_c,geonameid,city,latitude,longitude,population,date,pm25
dtype,object,object,object,object,string[python],Int64,object,float64,float64,int64,object,float64


In [6]:
pd.Series({'places': raw['geonameid'].nunique(), 'first date': raw['date'].min(), 'last date': raw['date'].max(),
           'distinct dates': raw['date'].nunique(), 'rows per place (min)': raw.groupby('geonameid').size().min(),
           'rows per place (max)': raw.groupby('geonameid').size().max(), 'null cells': int(raw.isna().sum().sum())})

places                         339
first date              2018-10-01
last date               2026-08-30
distinct dates                2797
rows per place (min)          2797
rows per place (max)          2797
null cells                       0
dtype: object

In [7]:
raw['adm1cd_c'].value_counts().rename('places x days')

adm1cd_c
PAK006    523039
PAK007    229354
PAK005    114677
PAK002     75519
PAK003      5594
Name: places x days, dtype: Int64

## Kaggle / Hugging Face hourly file

In [8]:
hf_raw = load.load_hf_raw()
print(hf_raw.shape)
hf_raw.head(3)

(21840, 26)


,timestamp,city,latitude,longitude,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,temperature,humidity,precipitation,wind_speed,wind_direction,pressure,date,hour,day_of_week,month,month_name,year,is_weekend,season,aqi_category
0,2025-11-06 00:00:00,Faisalabad,31.4504,73.135,23.7,22.6,598.0,15.9,5.0,58.0,2.0,16.8,70.0,0.0,1.0,45.0,1023.4,2025-11-06,0,Thursday,11,November,2025,0,Autumn,Moderate
1,2025-11-06 01:00:00,Faisalabad,31.4504,73.135,25.4,24.2,790.0,21.7,5.6,51.0,2.0,16.8,70.0,0.0,1.0,45.0,1023.4,2025-11-06,1,Thursday,11,November,2025,0,Autumn,Moderate
2,2025-11-06 02:00:00,Faisalabad,31.4504,73.135,32.2,30.7,1048.0,29.4,6.5,42.0,2.0,16.8,70.0,0.0,1.0,45.0,1023.4,2025-11-06,2,Thursday,11,November,2025,0,Autumn,Moderate


In [9]:
hf_raw.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
latitude,21840.0,31.04,2.67,24.86,30.16,31.50,33.57,34.02
longitude,21840.0,71.54,2.58,66.97,70.30,72.27,73.14,74.52
pm10,21840.0,88.42,66.01,0.50,40.50,68.50,117.80,488.00
pm2_5,21840.0,82.97,65.74,0.50,34.70,62.40,112.50,486.20
carbon_monoxide,21840.0,1580.86,1145.63,91.00,741.00,1296.50,2113.25,11482.00
nitrogen_dioxide,21840.0,39.47,32.29,0.00,13.40,30.40,57.80,185.10
sulphur_dioxide,21840.0,19.47,11.58,0.10,10.60,17.70,26.30,75.10
ozone,21840.0,63.80,50.40,0.00,21.00,55.00,100.00,265.00
dust,21840.0,9.86,32.53,0.00,1.00,3.00,8.00,711.00
temperature,21840.0,13.72,5.04,-5.90,10.20,13.30,16.90,30.40


In [10]:
hf_raw.groupby('city').agg(rows=('timestamp','size'), start=('timestamp','min'), end=('timestamp','max'))

,rows,start,end
city,,,
Faisalabad,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Islamabad,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Karachi,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Lahore,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Multan,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Peshawar,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Quetta,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Rahim Yar Khan,2184,2025-11-06 00:00:00,2026-02-04 23:00:00
Rawalpindi,2184,2025-11-06 00:00:00,2026-02-04 23:00:00


## Open-Meteo ERA5 weather payloads

In [11]:
payloads, wman = load.load_weather_raw()
print(len(payloads), 'cities;', wman.get('model'), wman.get('timezone'))
pd.DataFrame([{**{'geonameid': p['geonameid']}, **p['meta']} for p in payloads])[['city','reason','requested_lat','grid_lat','requested_lon','grid_lon']].head(25)

22 cities; era5 Asia/Karachi


,city,reason,requested_lat,grid_lat,requested_lon,grid_lon
0,Lahore,top_population,31.55800,31.50,74.35071,74.25
1,Peshawar,top_population,34.00800,34.00,71.57849,71.50
2,Faisalabad,top_population,31.41554,31.50,73.08969,73.00
3,Rawalpindi,top_population,33.59733,33.50,73.04790,73.00
4,Gujranwala,top_population,32.15567,32.25,74.18705,74.25
5,Multan,top_population,30.19679,30.25,71.47824,71.50
6,Hyderabad,top_population,25.39689,25.50,68.37718,68.50
7,Quetta,top_population,30.18414,30.25,67.00141,67.00
8,Bannu,top_population,32.98527,33.00,70.60403,70.50
9,Sargodha,top_population,32.08586,32.00,72.67418,72.75


## Coverage summary
Each source is kept in its own analytical table.

In [12]:
from src.utils.config import SOURCES
cov = dq.source_coverage({'wb_pm25': (wb, 'date'), 'hf_aq': (load.load_processed('hf_hourly'), 'timestamp'),
                          'openmeteo_era5': (load.load_processed('weather_daily'), 'date')}, SOURCES)
cov

,source_id,source,value_type,resolution,license,cities,start,end,rows
0,wb_pm25,World Bank PM2.5 (modelled),modelled,daily,CC BY-SA 4.0,339,2018-10-01,2026-08-30,948183
1,hf_aq,"Kaggle/HF hourly AQ (modelled, probable CAMS)",modelled,hourly,Unknown (not stated by uploader),10,2025-11-06,2026-02-04,21840
2,openmeteo_era5,Open-Meteo ERA5 weather (reanalysis),modelled,daily,CC BY 4.0 (Open-Meteo API data); ERA5 from Cop...,22,2018-10-01,2026-08-30,63602


In [13]:
charts.source_timeline(cov)